In [0]:
%pip install faker

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import json
from pyspark.sql.functions import col, udf, from_json, current_timestamp
from pyspark.sql.types import StructType, StructField, LongType, StringType, IntegerType, FloatType
from faker import Faker

# Shared pool of car_numbers for referential integrity
CAR_NUMBERS = [
    "ABC-1234", "XYZ-5678", "DEF-9012", "GHI-3456", "JKL-7890",
    "MNO-2345", "PQR-6789", "STU-0123", "VWX-4567", "YZA-8901",
    "BCD-2345", "EFG-6789", "HIJ-0123", "KLM-4567", "NOP-8901",
    "QRS-2345", "TUV-6789", "WXY-0123", "ZAB-4567", "CDE-8901",
    "FGH-2345", "IJK-6789", "LMN-0123", "OPQ-4567", "RST-8901",
    "UVW-2345", "XYZ-6789", "ABC-0123", "DEF-4567", "GHI-8901",
    "JKL-2345", "MNO-6789", "PQR-0123", "STU-4567", "VWX-8901",
    "YZA-2345", "BCD-6789", "EFG-0123", "HIJ-4567", "KLM-8901",
    "NOP-2345", "QRS-6789", "TUV-0123", "WXY-4567", "ZAB-8901",
    "CDE-2345", "FGH-6789", "IJK-0123", "LMN-4567", "OPQ-8901"
]

# UDFs to Generate Fake JSON Data
@udf(returnType=StringType())
def generate_driver_json(val):
    fake = Faker()
    car_number = CAR_NUMBERS[fake.random_int(0, len(CAR_NUMBERS) - 1)]
    return json.dumps({"id": val, "name": fake.name(), "car_number": car_number, "experience": fake.random_int(min=1, max=40), "rating": round(fake.pyfloat(min_value=1.0, max_value=5.0), 1)})

@udf(returnType=StringType())
def generate_vehicle_json(val):
    fake = Faker()
    car_number = CAR_NUMBERS[val % len(CAR_NUMBERS)]
    return json.dumps({"car_number": car_number, "model": fake.word().capitalize(), "year": fake.random_int(min=2010, max=2024), "status": "Active"})

@udf(returnType=StringType())
def generate_trip_json(val):
    fake = Faker()
    car_number = CAR_NUMBERS[fake.random_int(0, len(CAR_NUMBERS) - 1)]
    return json.dumps({"trip_id": f"T-{val}", "driver_id": val, "car_number": car_number, "distance_miles": round(fake.pyfloat(min_value=1.0, max_value=50.0), 1), "fare_amount": round(fake.pyfloat(min_value=5.0, max_value=100.0), 2)})
  


# Bronze Streams
rate_stream = spark.readStream.format("rate").option("rowsPerSecond", 1).load()

# Stream 1: Drivers
rate_stream.withColumn("raw_payload", generate_driver_json(col("value"))).select("timestamp", "raw_payload") \
    .writeStream.format("delta").outputMode("append").option("checkpointLocation", "/Volumes/logistics_lakehouse/bronze/checkpoints/bronze_drivers") \
    .trigger(availableNow=True).toTable("logistics_lakehouse.bronze.raw_drivers").awaitTermination(timeout=120)

# Stream 2: Vehicles
rate_stream.withColumn("raw_payload", generate_vehicle_json(col("value"))).select("timestamp", "raw_payload") \
    .writeStream.format("delta").outputMode("append").option("checkpointLocation", "/Volumes/logistics_lakehouse/bronze/checkpoints/bronze_vehicles") \
    .trigger(availableNow=True).toTable("logistics_lakehouse.bronze.raw_vehicles").awaitTermination(timeout=120)

# Stream 3: Trips
rate_stream.withColumn("raw_payload", generate_trip_json(col("value"))).select("timestamp", "raw_payload") \
    .writeStream.format("delta").outputMode("append").option("checkpointLocation", "/Volumes/logistics_lakehouse/bronze/checkpoints/bronze_trips") \
    .trigger(availableNow=True).toTable("logistics_lakehouse.bronze.raw_trips").awaitTermination(timeout=120)


True

In [0]:
# Silver Streams
driver_schema = StructType([StructField("id", LongType()), StructField("name", StringType()), StructField("car_number", StringType()), StructField("experience", IntegerType()), StructField("rating", FloatType())])
vehicle_schema = StructType([StructField("car_number", StringType()), StructField("model", StringType()), StructField("year", IntegerType()), StructField("status", StringType())])
trip_schema = StructType([
    StructField("trip_id", StringType()), 
    StructField("driver_id", LongType()), 
    StructField("car_number", StringType()),
    StructField("distance_miles", FloatType()), 
    StructField("fare_amount", FloatType())
])

# Parse Drivers
spark.readStream.format("delta").table("logistics_lakehouse.bronze.raw_drivers") \
    .withColumn("parsed", from_json(col("raw_payload"), driver_schema)).select("parsed.*", "timestamp") \
    .writeStream.format("delta").outputMode("append").option("checkpointLocation", "/Volumes/logistics_lakehouse/bronze/checkpoints/silver_drivers") \
    .trigger(availableNow=True).toTable("logistics_lakehouse.silver.drivers").awaitTermination(timeout=120)

# Parse Vehicles
spark.readStream.format("delta").table("logistics_lakehouse.bronze.raw_vehicles") \
    .withColumn("parsed", from_json(col("raw_payload"), vehicle_schema)).select("parsed.*", "timestamp") \
    .writeStream.format("delta").outputMode("append").option("checkpointLocation", "/Volumes/logistics_lakehouse/bronze/checkpoints/silver_vehicles") \
    .trigger(availableNow=True).toTable("logistics_lakehouse.silver.vehicles").awaitTermination(timeout=120)

# Parse Trips
spark.readStream.format("delta").table("logistics_lakehouse.bronze.raw_trips") \
    .withColumn("parsed", from_json(col("raw_payload"), trip_schema)).select("parsed.*", "timestamp") \
    .writeStream.format("delta").outputMode("append").option("checkpointLocation", "/Volumes/logistics_lakehouse/bronze/checkpoints/silver_trips") \
    .trigger(availableNow=True).toTable("logistics_lakehouse.silver.trips").awaitTermination(timeout=120)

True

In [0]:
# Part about Schema Evolution

# Update Bronze UDF to include Phone
@udf(returnType=StringType())
def generate_driver_json_v2(val):
    fake = Faker()
    car_number = CAR_NUMBERS[fake.random_int(0, len(CAR_NUMBERS) - 1)]
    return json.dumps({"id": val, "name": fake.name(), "car_number": car_number, "experience": fake.random_int(min=1, max=40), "rating": round(fake.pyfloat(min_value=1.0, max_value=5.0), 1), "phone": fake.phone_number()})

spark.readStream.format("rate").option("rowsPerSecond", 1).load() \
    .withColumn("raw_payload", generate_driver_json_v2(col("value"))).select("timestamp", "raw_payload") \
    .writeStream.format("delta").outputMode("append").option("checkpointLocation", "/Volumes/logistics_lakehouse/bronze/checkpoints/bronze_drivers") \
    .trigger(availableNow=True).toTable("logistics_lakehouse.bronze.raw_drivers").awaitTermination(timeout=120)

# Update Silver Schema with MergeSchema enabled
driver_schema_v2 = StructType([StructField("id", LongType()), StructField("name", StringType()), StructField("car_number", StringType()), StructField("experience", IntegerType()), StructField("rating", FloatType()), StructField("phone", StringType(), True)])

spark.readStream.format("delta").table("logistics_lakehouse.bronze.raw_drivers") \
    .withColumn("parsed", from_json(col("raw_payload"), driver_schema_v2)).select("parsed.*", "timestamp") \
    .writeStream.format("delta").outputMode("append").option("checkpointLocation", "/Volumes/logistics_lakehouse/bronze/checkpoints/silver_drivers") \
    .option("mergeSchema", "true") \
    .trigger(availableNow=True).toTable("logistics_lakehouse.silver.drivers").awaitTermination(timeout=120)

True

In [0]:
%sql
DESCRIBE HISTORY logistics_lakehouse.silver.drivers;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
16,2026-07-14T22:56:26.000Z,77600554195762,temur.tsomaia@innowise.com,STREAMING UPDATE,"Map(outputMode -> Append, queryId -> c8f09fa9-b012-46cf-86dd-7496d12ce464, epochId -> 3, statsOnLoad -> false)",null,List(964873248568909),ff409207-f65b-4a4c-81c1-4df93879d743,0714-180459-yh7nqatu-v2n,15,WriteSerializable,true,"Map(numRemovedFiles -> 0, numOutputRows -> 22, numOutputBytes -> 3081, numAddedFiles -> 1)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
15,2026-07-14T22:56:11.000Z,77600554195762,temur.tsomaia@innowise.com,STREAMING UPDATE,"Map(outputMode -> Append, queryId -> c8f09fa9-b012-46cf-86dd-7496d12ce464, epochId -> 2, statsOnLoad -> false)",null,List(964873248568909),605f6d06-0ecc-461b-801a-e5273db771d0,0714-180459-yh7nqatu-v2n,14,WriteSerializable,true,"Map(numRemovedFiles -> 0, numOutputRows -> 66, numOutputBytes -> 3315, numAddedFiles -> 1)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
14,2026-07-14T22:55:07.000Z,77600554195762,temur.tsomaia@innowise.com,STREAMING UPDATE,"Map(outputMode -> Append, queryId -> c8f09fa9-b012-46cf-86dd-7496d12ce464, epochId -> 1, statsOnLoad -> false)",null,List(964873248568909),76a7eb5e-6642-4d4f-9c40-136ff3d152d8,0714-180459-yh7nqatu-v2n,13,WriteSerializable,true,"Map(numRemovedFiles -> 0, numOutputRows -> 597, numOutputBytes -> 14064, numAddedFiles -> 1)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
13,2026-07-14T22:47:37.000Z,77600554195762,temur.tsomaia@innowise.com,STREAMING UPDATE,"Map(outputMode -> Append, queryId -> c8f09fa9-b012-46cf-86dd-7496d12ce464, epochId -> 0, statsOnLoad -> false)",null,List(964873248568909),6cb85a60-5537-4dd8-afaf-be07d514426a,0714-180459-yh7nqatu-v2n,12,WriteSerializable,true,"Map(numRemovedFiles -> 0, numOutputRows -> 490, numOutputBytes -> 7489, numAddedFiles -> 1)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
12,2026-07-14T22:41:46.000Z,77600554195762,temur.tsomaia@innowise.com,DELETE,"Map(predicate -> [""true""])",null,List(964873248568909),6478c4a7-9ed2-4358-ad96-08414178f926,0714-180459-yh7nqatu-v2n,11,WriteSerializable,false,"Map(numRemovedFiles -> 6, numRemovedBytes -> 38964, numCopiedRows -> 0, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 7, numDeletionVectorsUpdated -> 0, numDeletedRows -> 1728, scanTimeMs -> 7, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 0)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
11,2026-07-14T22:02:30.000Z,77600554195762,temur.tsomaia@innowise.com,STREAMING UPDATE,"Map(outputMode -> Append, queryId -> 4eaf2b0f-af46-462b-90d6-2219e9c46a7b, epochId -> 7, statsOnLoad -> false)",null,List(964873248568909),b2ccec91-a380-4061-b675-f3c979bf63aa,0714-180459-yh7nqatu-v2n,10,WriteSerializable,true,"Map(numRemovedFiles -> 0, numOutputRows -> 35, numOutputBytes -> 3486, numAddedFiles -> 1)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
10,2026-07-14T22:02:14.000Z,77600554195762,temur.tsomaia@innowise.com,STREAMING UPDATE,"Map(outputMode -> Append, queryId -> 4eaf2b0f-af46-462b-90d6-2219e9c46a7b, epochId -> 6, statsOnLoad -> false)",null,List(964873248568909),8092d510-bec2-4913-aecc-c5cf1d6c842f,0714-180459-yh7nqatu-v2n,9,WriteSerializable,true,"Map(numRemovedFiles -> 0, numOutputRows -> 605, numOutputBytes -> 10653, numAddedFiles -> 1)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
9,2026-07-14T21:51:50.000Z,77600554195762,temur.tsomaia@innowise.com,STREAMING UPDATE,"Map(outputMode -> Append, queryId -> 4eaf2b0f-af46-462b-90d6-2219e9c46a7b, epochId -> 5, statsOnLoad -> false)",null,List(964873248568909),dd30f4b4-f739-43de-bcd4-399762911635,0714-180459-yh7nqatu-v2n,8,WriteSerializable,true,"Map(numRemovedFiles -> 0, numOutputRows -> 34, numOutputBytes -> 3423, numAddedFiles -> 1)",null,Databricks-Runtime/18.x-a

In [0]:
%sql
-- Retrieve records as they appeared "yesterday"
-- SELECT * FROM logistics_lakehouse.silver.drivers TIMESTAMP AS OF (current_timestamp() - INTERVAL 1 DAY);

-- View Version 0 (before phone was added) and Restore it
SELECT * FROM logistics_lakehouse.silver.drivers VERSION AS OF 5;
-- RESTORE TABLE logistics_lakehouse.silver.drivers TO VERSION AS OF 0; --Optional

id,name,car_number,experience,rating,timestamp
2,Richard Figueroa,J 266196,11,4.1,2026-07-14T18:32:34.757Z
10,Billy Miller,7296 SN,4,4.1,2026-07-14T18:32:42.757Z
18,Nicole Ford,U32 4DK,35,1.4,2026-07-14T18:32:50.757Z
1,Dana Stephenson,DJB-0244,10,3.2,2026-07-14T18:32:33.757Z
9,Victoria Lane,0OTJ220,8,1.5,2026-07-14T18:32:41.757Z
17,Victoria Williams,JMZ 465,38,2.3,2026-07-14T18:32:49.757Z
0,Frank Stevenson,BVY-3313,4,1.3,2026-07-14T18:32:32.757Z
8,Megan Jackson,7G 7807S,15,1.3,2026-07-14T18:32:40.757Z
16,Shelia Smith,FVT 350,16,4.6,2026-07-14T18:32:48.757Z
7,Mrs. Ann Nichols,KXU-530,28,1.7,2026-07-14T18:32:39.757Z


In [0]:
%sql
SELECT * FROM logistics_lakehouse.silver.drivers WHERE phone IS NOT NULL

id,name,car_number,experience,rating,timestamp,phone
0,James Mitchell,STU-4567,2,3.7,2026-07-14T22:44:55.162Z,(538)424-1367
8,Angela Arnold,YZA-2345,15,4.7,2026-07-14T22:45:03.162Z,436-689-2479
16,Lee King,XYZ-6789,13,1.6,2026-07-14T22:45:11.162Z,(445)209-6563x395
24,Scott Swanson,ABC-1234,18,1.4,2026-07-14T22:45:19.162Z,(619)640-6305x60108
32,Ashley Hudson,ABC-1234,30,1.6,2026-07-14T22:45:27.162Z,655.740.1166x934
40,Michael Jones,IJK-0123,12,3.6,2026-07-14T22:45:35.162Z,894.323.3962
48,Gail Patton,TUV-6789,4,1.9,2026-07-14T22:45:43.162Z,+1-205-782-8700x087
56,Bradley Schroeder,RST-8901,4,1.8,2026-07-14T22:45:51.162Z,687.444.8852
64,Ashley White,HIJ-0123,16,3.8,2026-07-14T22:45:59.162Z,452-806-0072x758
72,Laura Morgan,OPQ-4567,32,2.8,2026-07-14T22:46:07.162Z,864.418.3089


In [0]:
%sql
-- Transforming Silver Data into Gold/Populating (DML)
INSERT INTO logistics_lakehouse.gold.dim_drivers
SELECT 
    id AS driver_id, 
    name, 
    experience, 
    CASE 
        WHEN experience >= 15 THEN 'Senior'
        WHEN experience >= 5 THEN 'Mid-Level'
        ELSE 'Junior'
    END AS experience_level,
    rating, 
    CASE 
        WHEN rating >= 4.5 THEN True 
        ELSE False 
    END AS is_premium,
    phone,
    current_timestamp() AS loaded_at
FROM logistics_lakehouse.silver.drivers;

INSERT INTO logistics_lakehouse.gold.dim_vehicles
SELECT 
    car_number, 
    model, 
    year, 
    CASE 
        WHEN year >= 2020 THEN 'Modern Fleet'
        ELSE 'Legacy Fleet'
    END AS vehicle_age_category,
    status,
    current_timestamp() AS loaded_at
FROM logistics_lakehouse.silver.vehicles;

INSERT INTO logistics_lakehouse.gold.fact_trips
SELECT 
    trip_id, 
    driver_id, 
    car_number, 
    distance_miles, 
    fare_amount,
    ROUND(fare_amount / distance_miles, 2) AS fare_per_mile,
    current_timestamp() AS loaded_at
FROM logistics_lakehouse.silver.trips;


-- Optimization
OPTIMIZE logistics_lakehouse.silver.trips ZORDER BY (trip_id);
OPTIMIZE logistics_lakehouse.gold.dim_drivers;
OPTIMIZE logistics_lakehouse.gold.fact_trips;


path,metrics
,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 1, 0, false, 0, 0, 1784069807604, 1784069810193, 8, 0, null, List(0, 0), null, 7, 7, 0, 0, List(17165, true, false, false, null, null, null, null, 0, 0, 0, 0, 1, 17165, 17165, null, log, 16777216, 67108864, 4, 0, 0, null, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, List(74, 5, 0, 0, 0, 1230), 2, 1, 5, sizeAware, defaultSplitStrategy, null, false, 0, null, false, 0, 0, 0, null, null, null), null)"
,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 1, 1, true, 0, 0, 1784069810226, 1784069811849, 8, 0, null, List(0, 0), null, 7, 7, 0, 0, List(17165, false, false, false, null, null, null, post-optimize-compaction, 0, 0, 0, 0, 0, 0, 0, null, null, 33554432, 67108864, 0, 0, 0, null, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, List(0, 0, 771, 0, 0, 0), 15, 1, 1, null, null, null, false, 0, null, false, 0, 0, 0, null, null, null), null)"
